In [1]:
# importing major libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('credit_card_fraud.csv').drop('isFlaggedFraud',axis=1)

In [3]:
df

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0
...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,1


In [4]:
df.dropna().isFraud

0          0
1          0
2          1
3          1
4          0
          ..
6362615    1
6362616    1
6362617    1
6362618    1
6362619    1
Name: isFraud, Length: 6362620, dtype: int64

In [5]:
# unwanted columns drop
cols_to_drop = ['nameOrig', 'nameDest']
df.drop(cols_to_drop, axis=1, inplace=True)

In [6]:
# check categorical column
df['type'].unique()

# features & target
X = df.iloc[:,0 :-1]
y = df['isFraud']

In [7]:
# train test split
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# ColumnTransformer using df.columns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

cat_cols = df.select_dtypes(include='object').columns

trans1 = ColumnTransformer(
    transformers=[
        ('encode', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
    ],
    remainder='passthrough'
)

In [9]:
# transform
x_train_encode = trans1.fit_transform(x_train)
x_test_encode = trans1.transform(x_test)

In [10]:
# convert to dataframe with column names
x_train_encode = pd.DataFrame(x_train_encode, columns=trans1.get_feature_names_out())
x_test_encode = pd.DataFrame(x_test_encode, columns=trans1.get_feature_names_out())

In [11]:
y_train.value_counts()

isFraud
0    5083503
1       6593
Name: count, dtype: int64

In [12]:
# solution undersampling
# oversampling

#--> undersampling

In [13]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler()
rus

RandomUnderSampler()

In [14]:
x_train_undersampled,y_train_undersampled = rus.fit_resample(x_train_encode,
y_train)

In [15]:
x_train_undersampled
y_train_undersampled.value_counts()

isFraud
0    6593
1    6593
Name: count, dtype: int64

In [16]:
from imblearn.over_sampling import RandomOverSampler

ros=RandomOverSampler()
ros

RandomOverSampler()

In [17]:
x_train_oversample,y_train_oversample = ros.fit_resample(x_train_encode,
y_train)                                                         

In [18]:
y_train_oversample.value_counts()

isFraud
0    5083503
1    5083503
Name: count, dtype: int64

In [19]:
# smote --> oversampling
from imblearn.over_sampling import SMOTE
smote = SMOTE()
smote

SMOTE()

In [20]:
x_train_smote,y_train_smote = smote.fit_resample(x_train_encode,y_train)

In [21]:
x_train_smote
y_train_smote

0           0
1           0
2           0
3           0
4           0
           ..
10167001    1
10167002    1
10167003    1
10167004    1
10167005    1
Name: isFraud, Length: 10167006, dtype: int64

In [22]:
# x_train,y_train
# x_train_encode,y_train
# x_train_undersample,y_train_undersample
# x_train_oversample,y_train_oversample
#x_train_smote,y_train_smote

In [23]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
trans = ColumnTransformer([
        ('encode', OneHotEncoder(sparse_output=False,dtype=int), ['type'])
    ],remainder='passthrough')
trans
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
pipe = Pipeline([
    ('tranformer',trans),
    ('smote',SMOTE()),
    ('model',LogisticRegression())
])
pipe.fit(x_train,y_train)

Pipeline(steps=[('tranformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('encode',
                                                  OneHotEncoder(dtype=<class 'int'>,
                                                                sparse_output=False),
                                                  ['type'])])),
                ('smote', SMOTE()), ('model', LogisticRegression())])

In [24]:
y_pred=pipe.predict(x_test)
y_pred

array([0, 1, 0, ..., 0, 0, 0])

In [25]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

In [26]:
accuracy_score(y_test, y_pred)

0.9175190408982463